# Create Work Parents (oxjob #2065)

Which works each work is part of. A chapter is part of its book, a figure of its paper, a file of its dataset.
`CreateWorksEnriched` puts three fields on the work from the last table here:

- `parent_works` `[{id, display_name, relationship}]`: the works this one is part of. `relationship` is one of
  `volume` (in a book or proceedings volume), `component` (a figure, table, file or supplement), `data-series` (one
  record of a data series: a NIFS plasma-shot record under its instrument series), `part` (any other declared part).
- `lineage`: the work, then its ancestors nearest first (as for institutions and publishers).
- `child_works_count`: how many works name this one as a parent.

```
openalex_works_base.volume (CreateWorkVolumes, #1554)     \
datacite_works.ids (IsPartOf -> DOI) + location_work_ids   --> work_parent_candidates
NIFS shot DOIs + the minted series works (location_work_ids) /
"Additional file N of <paper>" titles (openalex_works_base) /
work_parent_candidates   --> work_parent_edges_refused (more than 10 parents, loops, more than 10 generations)
                         --> work_parent_edges --> work_parent_ancestors --> work_parents --> CreateWorksEnriched
```

Runs after `Works_Base` (it needs tonight's live works, their types and titles) and before `Works_Enriched`.
Only part-of links: versions and copies are merges, citations are references. Spec, sizes and the accuracy bar:
oxjobs `working/work-parent-child-links/` (PLAN.md "DECIDED", EXPLORE.md, ACCEPTANCE.md Test 1).

### Candidate edges: every part-of link between two live works

In [0]:
-- work_parent_candidates (oxjob #2065): one row per (child, parent) pair of live works.
--   crossref_volume      the chapter -> book link CreateWorksBase already put on the work as `volume` (#1554)
--   datacite_is_part_of  a DataCite record that declares IsPartOf another DOI; both DOIs are mapped to their works
--                        through location_work_ids (the Crossref record's work when both registries hold the DOI)
-- >>> oxjob #2065 steps 3 + 5 (begin)
--   nifs_data_series     a NIFS plasma-shot record (DataCite 10.57451/lhd.<diagnostic>.<shot>.<n>, or lhd.a.<...> analyzed)
--                        -> the data-series work OpenAlex mints for its instrument (CreateLocationsWithTypes, provenance
--                        `openalex-series`); relationship `data-series`
--   nifs_lhd             each minted NIFS series work -> the minted "LHD experiment data" work; relationship `part`
--   figshare_title       a work titled "Additional file N of <paper title>" (BMC supplements on Figshare) -> the one
--                        traditional work with exactly that title (letters and digits compared); relationship `component`.
--                        No link when two or more works carry the title, or none.
-- <<< oxjob #2065 steps 3 + 5 (end)
-- A pair both sources name is one edge with relationship `volume`. A DataCite edge is `component` when the child is
-- typed supplementary-materials, other, peer-review or software, else `part` (the type as CreateWorksBase set it;
-- Works_Enriched's work-types cell may still change the served type of a few).
-- A pair whose two DOIs sit on the same work (merged files, versions) is no edge.
CREATE OR REPLACE TABLE openalex.works.work_parent_candidates
CLUSTER BY (child_work_id)
AS
WITH base AS (
  SELECT id, type, volume FROM openalex.works.openalex_works_base
),
volume_edges AS (
  SELECT id AS child_work_id, CAST(REGEXP_EXTRACT(volume.id, 'W([0-9]+)$', 1) AS BIGINT) AS parent_work_id
  FROM base WHERE volume IS NOT NULL
),
declared AS (
  SELECT DISTINCT LOWER(w.native_id) AS child_doi,
    LOWER(REGEXP_REPLACE(TRIM(i.id), '^(https?://(dx\\.)?doi\\.org/|doi:)', '')) AS parent_doi
  FROM openalex.datacite.datacite_works w LATERAL VIEW EXPLODE(w.ids) x AS i
  WHERE LOWER(i.relationship) = 'ispartof' AND LOWER(i.namespace) = 'doi'
),
doi_works AS (
  SELECT native_id AS doi, MAX_BY(work_id, provenance = 'crossref') AS work_id
  FROM openalex.works.location_work_ids
  WHERE native_id_namespace = 'doi' AND provenance IN ('crossref', 'datacite') AND work_id IS NOT NULL
    AND native_id IN (SELECT child_doi FROM declared UNION SELECT parent_doi FROM declared)
  GROUP BY native_id
),
datacite_edges AS (
  SELECT DISTINCT c.work_id AS child_work_id, p.work_id AS parent_work_id
  FROM declared d
  JOIN doi_works c ON c.doi = d.child_doi
  JOIN doi_works p ON p.doi = d.parent_doi
  WHERE c.work_id <> p.work_id
),
-- >>> oxjob #2065 step 3 (begin): NIFS data series. The minted records' pins give each series key its work; a shot
-- record's DOI names its series (the DOI minus shot number and counter). Daily schedules (lhd.p.daily.*), daily reports
-- (lhd.r.daily.*) and nifs.aurora-project do not match the shot pattern and get no parent.
nifs_series AS (
  SELECT native_id AS series_key, work_id
  FROM openalex.works.location_work_ids
  WHERE provenance = 'openalex-series' AND native_id_namespace = 'series' AND native_id LIKE 'nifs:lhd%' AND work_id IS NOT NULL
),
nifs_series_edges AS (
  SELECT DISTINCT r.work_id AS child_work_id, s.work_id AS parent_work_id
  FROM openalex.works.location_work_ids r
  JOIN nifs_series s
    ON s.series_key = CONCAT('nifs:lhd.', REGEXP_EXTRACT(LOWER(r.native_id), '^10\\.57451/lhd\\.((a\\.)?[^.]+)\\.[0-9]+\\.[0-9]+$', 1))
  WHERE r.provenance = 'datacite' AND r.native_id_namespace = 'doi' AND r.native_id LIKE '10.57451/lhd.%'
    AND r.work_id IS NOT NULL
),
nifs_lhd_edges AS (
  SELECT s.work_id AS child_work_id, top.work_id AS parent_work_id
  FROM nifs_series s JOIN nifs_series top ON top.series_key = 'nifs:lhd'
  WHERE s.series_key <> 'nifs:lhd'
),
-- <<< oxjob #2065 step 3 (end)
-- >>> oxjob #2065 step 5 (begin): Figshare supplements to their paper, by title. The paper must be a traditional type
-- (common.work_types.is_traditional) and the only such work whose whole title, letters and digits, equals the text after
-- "Additional file N of|for|to"; titles of 20 characters or fewer are never matched.
titled AS (
  SELECT id, type, title FROM openalex.works.openalex_works_base WHERE title IS NOT NULL
),
supplement_titles AS (
  SELECT id, LOWER(REGEXP_REPLACE(REGEXP_EXTRACT(title, '^(?i)additional file \\S+ (of|for|to) (.+)$', 2), '[^A-Za-z0-9]', '')) AS paper_key
  FROM titled WHERE title RLIKE '^(?i)additional file \\S+ (of|for|to) (.+)$'
),
paper_titles AS (
  SELECT LOWER(REGEXP_REPLACE(b.title, '[^A-Za-z0-9]', '')) AS paper_key, MIN(b.id) AS work_id
  FROM titled b
  WHERE b.type IN (SELECT id FROM openalex.common.work_types WHERE is_traditional)
    AND NOT b.title RLIKE '^(?i)additional file \\S+ (of|for|to) (.+)$'
    AND LOWER(REGEXP_REPLACE(b.title, '[^A-Za-z0-9]', '')) IN (SELECT paper_key FROM supplement_titles WHERE LENGTH(paper_key) > 20)
  GROUP BY 1
  HAVING COUNT(*) = 1
),
figshare_title_edges AS (
  SELECT s.id AS child_work_id, p.work_id AS parent_work_id
  FROM supplement_titles s JOIN paper_titles p ON p.paper_key = s.paper_key
),
-- <<< oxjob #2065 step 5 (end)
unioned AS (
  SELECT child_work_id, parent_work_id, 'crossref_volume' AS source FROM volume_edges
  UNION ALL
  SELECT child_work_id, parent_work_id, 'datacite_is_part_of' AS source FROM datacite_edges
  UNION ALL  -- oxjob #2065 step 3
  SELECT child_work_id, parent_work_id, 'nifs_data_series' AS source FROM nifs_series_edges
  UNION ALL  -- oxjob #2065 step 3
  SELECT child_work_id, parent_work_id, 'nifs_lhd' AS source FROM nifs_lhd_edges
  UNION ALL  -- oxjob #2065 step 5
  SELECT child_work_id, parent_work_id, 'figshare_title' AS source FROM figshare_title_edges
)
SELECT
  u.child_work_id,
  u.parent_work_id,
  MIN(u.source) AS source,
  CASE
    WHEN MIN(u.source) = 'crossref_volume' THEN 'volume'
    WHEN MIN(u.source) = 'nifs_data_series' THEN 'data-series'  -- oxjob #2065 step 3
    WHEN MIN(u.source) = 'nifs_lhd' THEN 'part'                 -- oxjob #2065 step 3
    WHEN MIN(u.source) = 'figshare_title' THEN 'component'      -- oxjob #2065 step 5 (a pair DataCite also declares keeps that source; its child is a supplement, so `component` too)
    WHEN ANY_VALUE(c.type) IN ('supplementary-materials', 'other', 'peer-review', 'software') THEN 'component'
    ELSE 'part'
  END AS relationship
FROM unioned u
JOIN base c ON c.id = u.child_work_id
JOIN base p ON p.id = u.parent_work_id
WHERE u.child_work_id <> u.parent_work_id
GROUP BY u.child_work_id, u.parent_work_id

### Refused edges: over-claiming children, loops, chains deeper than 10 generations

Three kinds of declared link are not a part-of hierarchy and are left out (kept in `work_parent_edges_refused` with
the reason):
- **more_than_10_parents**: a child that claims more than 10 parents (reversed `IsPartOf`: one OBIS dataset "is part
  of" 23,947 records). All its edges are refused.
- **loop**: an edge on a loop (two records that each say `IsPartOf` the other, or a longer ring).
- **more_than_10_generations**: a child with an ancestor more than 10 generations up (chains of unrelated preprints
  nested as "components" of one another). All its edges are refused; the institutions lineage has the same cap.

Only edges whose child is itself a parent can sit on a loop or a long chain (~37K of ~29.5M), so the walk below is over
that small graph: breadth first, one generation per pass, each (start, ancestor) pair kept once at its nearest
generation. A recursive CTE is not used: it stops at 1M rows.

In [0]:
-- The small graph: edges whose child is itself a parent, children claiming more than 10 parents left out.
CREATE OR REPLACE TABLE openalex.works.work_parent_internal_edges AS
WITH over_claiming AS (
  SELECT child_work_id FROM openalex.works.work_parent_candidates GROUP BY child_work_id HAVING COUNT(*) > 10
),
kept AS (
  SELECT c.child_work_id, c.parent_work_id FROM openalex.works.work_parent_candidates c
  LEFT ANTI JOIN over_claiming o ON o.child_work_id = c.child_work_id
)
SELECT k.child_work_id, k.parent_work_id FROM kept k
LEFT SEMI JOIN kept up ON up.parent_work_id = k.child_work_id

In [0]:
-- Walk up from every node of the small graph, 11 generations (one past the cap, to see who goes deeper).
BEGIN
  DECLARE g INT DEFAULT 1;
  DECLARE added BIGINT DEFAULT 1;
  CREATE OR REPLACE TABLE openalex.works.work_parent_reach AS
    SELECT child_work_id AS start_id, parent_work_id AS node_id, 1 AS generation FROM openalex.works.work_parent_internal_edges;
  WHILE added > 0 AND g < 11 DO
    INSERT INTO openalex.works.work_parent_reach
    SELECT DISTINCT r.start_id, e.parent_work_id, g + 1
    FROM openalex.works.work_parent_reach r
    JOIN openalex.works.work_parent_internal_edges e ON e.child_work_id = r.node_id
    LEFT ANTI JOIN openalex.works.work_parent_reach seen ON seen.start_id = r.start_id AND seen.node_id = e.parent_work_id
    WHERE r.generation = g;
    SET g = g + 1;
    SET added = (SELECT COUNT(*) FROM openalex.works.work_parent_reach WHERE generation = g);
  END WHILE;
END

In [0]:
CREATE OR REPLACE TABLE openalex.works.work_parent_edges_refused AS
WITH over_claiming AS (
  SELECT child_work_id FROM openalex.works.work_parent_candidates GROUP BY child_work_id HAVING COUNT(*) > 10
),
loops AS (
  -- child -> parent is on a loop when the child is also above the parent
  SELECT e.child_work_id, e.parent_work_id
  FROM openalex.works.work_parent_internal_edges e
  LEFT SEMI JOIN openalex.works.work_parent_reach r ON r.start_id = e.parent_work_id AND r.node_id = e.child_work_id
),
too_deep AS (
  SELECT DISTINCT start_id AS child_work_id FROM openalex.works.work_parent_reach WHERE generation > 10
)
SELECT c.child_work_id, c.parent_work_id, c.source, c.relationship,
  CASE WHEN o.child_work_id IS NOT NULL THEN 'more_than_10_parents'
       WHEN l.child_work_id IS NOT NULL THEN 'loop'
       ELSE 'more_than_10_generations' END AS reason
FROM openalex.works.work_parent_candidates c
LEFT JOIN over_claiming o ON o.child_work_id = c.child_work_id
LEFT JOIN loops l ON l.child_work_id = c.child_work_id AND l.parent_work_id = c.parent_work_id
LEFT JOIN too_deep d ON d.child_work_id = c.child_work_id
WHERE o.child_work_id IS NOT NULL OR l.child_work_id IS NOT NULL OR d.child_work_id IS NOT NULL

### The edges, and each parent's own ancestors

In [0]:
CREATE OR REPLACE TABLE openalex.works.work_parent_edges
CLUSTER BY (child_work_id)
AS
SELECT c.* FROM openalex.works.work_parent_candidates c
LEFT ANTI JOIN openalex.works.work_parent_edges_refused r
  ON r.child_work_id = c.child_work_id AND r.parent_work_id = c.parent_work_id

In [0]:
-- Ancestors of every work that is both a child and a parent, nearest generation kept, 10 generations at most.
BEGIN
  DECLARE g INT DEFAULT 1;
  DECLARE added BIGINT DEFAULT 1;
  CREATE OR REPLACE TABLE openalex.works.work_parent_ancestors AS
    SELECT e.child_work_id AS work_id, e.parent_work_id AS ancestor_work_id, 1 AS generation
    FROM openalex.works.work_parent_edges e
    LEFT SEMI JOIN openalex.works.work_parent_edges up ON up.parent_work_id = e.child_work_id;
  WHILE added > 0 AND g < 10 DO
    INSERT INTO openalex.works.work_parent_ancestors
    SELECT DISTINCT r.work_id, e.parent_work_id, g + 1
    FROM openalex.works.work_parent_ancestors r
    JOIN openalex.works.work_parent_edges e ON e.child_work_id = r.ancestor_work_id
    LEFT ANTI JOIN openalex.works.work_parent_ancestors seen ON seen.work_id = r.work_id AND seen.ancestor_work_id = e.parent_work_id
    WHERE r.generation = g AND e.parent_work_id <> r.work_id;
    SET g = g + 1;
    SET added = (SELECT COUNT(*) FROM openalex.works.work_parent_ancestors WHERE generation = g);
  END WHILE;
END

### One row per linked work: `parent_works`, `lineage`, `child_works_count`

In [0]:
-- work_parents (oxjob #2065): what CreateWorksEnriched merges onto the work. One row per work that has a parent or a
-- child. parent_works: the book first, then by id. display_name is the parent work's title tonight.
-- lineage: the work, then its ancestors by generation (nearest first), then id. A work with no parent: itself only.
-- Built in a staging table and swapped in only after the checks below pass.
CREATE OR REPLACE TABLE openalex.works.work_parents_staging
CLUSTER BY (work_id)
AS
WITH e AS (SELECT * FROM openalex.works.work_parent_edges),
titles AS (
  SELECT id, title FROM openalex.works.openalex_works_base WHERE id IN (SELECT parent_work_id FROM e)
),
parents AS (
  SELECT e.child_work_id AS work_id,
    TRANSFORM(
      ARRAY_SORT(COLLECT_LIST(NAMED_STRUCT('volume_first', IF(e.relationship = 'volume', 0, 1),
                                           'parent_work_id', e.parent_work_id,
                                           'display_name', t.title, 'relationship', e.relationship))),
      x -> NAMED_STRUCT('id', CONCAT('https://openalex.org/W', x.parent_work_id), 'display_name', x.display_name,
                        'relationship', x.relationship)) AS parent_works
  FROM e JOIN titles t ON t.id = e.parent_work_id
  GROUP BY e.child_work_id
),
ancestor_rows AS (
  SELECT child_work_id AS work_id, parent_work_id AS ancestor_work_id, 1 AS generation FROM e
  UNION ALL
  SELECT e.child_work_id, a.ancestor_work_id, a.generation + 1
  FROM e JOIN openalex.works.work_parent_ancestors a ON a.work_id = e.parent_work_id
  WHERE a.ancestor_work_id <> e.child_work_id
),
nearest AS (
  SELECT work_id, ancestor_work_id, MIN(generation) AS generation FROM ancestor_rows GROUP BY work_id, ancestor_work_id
),
lineages AS (
  SELECT work_id,
    TRANSFORM(ARRAY_SORT(COLLECT_LIST(NAMED_STRUCT('generation', generation, 'ancestor_work_id', ancestor_work_id))),
              x -> CONCAT('https://openalex.org/W', x.ancestor_work_id)) AS ancestors
  FROM nearest GROUP BY work_id
),
children AS (
  SELECT parent_work_id AS work_id, CAST(COUNT(*) AS INT) AS child_works_count FROM e GROUP BY parent_work_id
)
SELECT
  COALESCE(p.work_id, c.work_id) AS work_id,
  p.parent_works,
  CONCAT(ARRAY(CONCAT('https://openalex.org/W', COALESCE(p.work_id, c.work_id))), COALESCE(l.ancestors, ARRAY())) AS lineage,
  COALESCE(c.child_works_count, 0) AS child_works_count
FROM parents p
LEFT JOIN lineages l ON l.work_id = p.work_id
FULL OUTER JOIN children c ON c.work_id = p.work_id

### Checks, then publish

The night stops here, before `Works_Enriched`, if the build is not a hierarchy (a work listed twice, a work in its own
lineage twice, a child with more than 10 parents, a chapter whose book is missing) or if a source went missing
(fewer than 20M book links or 2M DataCite links would un-link millions of works and re-stamp them).

In [0]:
SELECT
  ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT work_id), 'work_parents: a work is listed twice'),
  ASSERT_TRUE(COUNT_IF(SIZE(lineage) <> SIZE(ARRAY_DISTINCT(lineage))) = 0, 'work_parents: a work is in a lineage twice'),
  ASSERT_TRUE(COUNT_IF(SIZE(parent_works) > 10) = 0, 'work_parents: a child with more than 10 parents'),
  ASSERT_TRUE(COUNT_IF(parent_works IS NOT NULL AND SIZE(lineage) < 2) = 0, 'work_parents: a child whose lineage lacks its parent'),
  ASSERT_TRUE(COUNT_IF(EXISTS(parent_works, x -> x.id = CONCAT('https://openalex.org/W', work_id))) = 0, 'work_parents: a work is its own parent'),
  ASSERT_TRUE(COUNT_IF(EXISTS(parent_works, x -> x.relationship = 'volume')) >= 20000000, 'work_parents: fewer than 20M works in a volume; is openalex_works_base.volume empty?'),
  ASSERT_TRUE(COUNT_IF(EXISTS(parent_works, x -> x.relationship <> 'volume')) >= 2000000, 'work_parents: fewer than 2M DataCite children; is datacite_works or location_work_ids short?'),
  ASSERT_TRUE(SUM(child_works_count) = SUM(COALESCE(SIZE(parent_works), 0)), 'work_parents: child counts and parent lists disagree')
FROM openalex.works.work_parents_staging

In [0]:
-- A work CreateWorksBase gave a volume keeps that book as a `volume` parent, unless its edges were refused above (a
-- chapter that also claims 10 DataCite parents, or sits on a loop) or the book is not a live work tonight. Those are
-- a handful (0 on 13 Nov 2026); more than 0.1% of the works in a volume means the build is wrong.
SELECT ASSERT_TRUE(COUNT_IF(p.work_id IS NULL OR NOT EXISTS(p.parent_works, x -> x.id = b.volume.id AND x.relationship = 'volume'))
                     <= 0.001 * COUNT(*),
                   'work_parents: more than 0.1% of the works with a volume do not list it as a parent') AS checked,
       COUNT_IF(p.work_id IS NULL OR NOT EXISTS(p.parent_works, x -> x.id = b.volume.id AND x.relationship = 'volume')) AS works_with_a_volume_and_no_such_parent,
       COUNT(*) AS works_with_a_volume
FROM openalex.works.openalex_works_base b
LEFT JOIN openalex.works.work_parents_staging p ON p.work_id = b.id
WHERE b.volume IS NOT NULL

In [0]:
-- NIFS data series (step 3): once the minted series works exist, every plasma-shot work must be under its series.
-- Shot works with no `data-series` parent would show 15.6M loose rows in default lists, and losing the link on 44M
-- works at once (the minted records missing from locations_w_types, the registry short) would re-stamp them all:
-- more than 1% without one stops the night. Before the series are minted (or after they are switched off) there is
-- nothing to check and this passes.
SELECT ASSERT_TRUE(MAX(s.series_works) = 0 OR COUNT_IF(p.work_id IS NULL) <= 0.01 * COUNT(*),
                   'work_parents: more than 1% of the NIFS shot works have no data-series parent') AS checked,
       COUNT_IF(p.work_id IS NULL) AS shot_works_with_no_series, COUNT(*) AS shot_works, MAX(s.series_works) AS series_works
FROM (
  SELECT DISTINCT r.work_id
  FROM openalex.works.location_work_ids r
  LEFT SEMI JOIN openalex.works.openalex_works_base b ON b.id = r.work_id
  WHERE r.provenance = 'datacite' AND r.native_id_namespace = 'doi' AND r.native_id LIKE '10.57451/lhd.%'
    AND LOWER(r.native_id) RLIKE '^10\\.57451/lhd\\.((a\\.)?[^.]+)\\.[0-9]+\\.[0-9]+$' AND r.work_id IS NOT NULL
) w
LEFT JOIN (SELECT work_id FROM openalex.works.work_parents_staging WHERE EXISTS(parent_works, x -> x.relationship = 'data-series')) p
  ON p.work_id = w.work_id
CROSS JOIN (
  SELECT COUNT(*) AS series_works
  FROM openalex.works.location_work_ids r LEFT SEMI JOIN openalex.works.openalex_works_base b ON b.id = r.work_id
  WHERE r.provenance = 'openalex-series' AND r.native_id_namespace = 'series' AND r.native_id LIKE 'nifs:lhd.%'
) s

In [0]:
CREATE OR REPLACE TABLE openalex.works.work_parents
CLUSTER BY (work_id)
AS SELECT * FROM openalex.works.work_parents_staging

In [0]:
-- Tonight's numbers, for the run page.
SELECT 'works with a parent' AS what, COUNT_IF(parent_works IS NOT NULL) AS n FROM openalex.works.work_parents
UNION ALL SELECT 'works with a child', COUNT_IF(child_works_count > 0) FROM openalex.works.work_parents
UNION ALL SELECT CONCAT('edges: ', relationship), COUNT(*) FROM openalex.works.work_parent_edges GROUP BY relationship
UNION ALL SELECT CONCAT('edges from ', source), COUNT(*) FROM openalex.works.work_parent_edges GROUP BY source  -- oxjob #2065 steps 3 + 5
UNION ALL SELECT CONCAT('refused: ', reason), COUNT(*) FROM openalex.works.work_parent_edges_refused GROUP BY reason